# SafeCityAI custom YOLOv5 — training and handover

This notebook is **unexecuted scaffolding**, not evidence of a trained model or accuracy.
Use Runtime → Change runtime type → GPU. Supply a licensed, annotated YOLO dataset
with `images/train`, `images/val`, `labels/train`, `labels/val`. Split source videos
before extracting frames to avoid leakage. Target IDs: 0 Helmet, 1 NoHelmet, 2 LicensePlate.

Publish/review the working branch before running the clone cell. Do not point this
notebook at the old ONNX-fallback `main` branch and assume it contains these fixes.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json

ROOT = Path("/content/safecityai")
REPO = "https://github.com/akshaydip11-source/object-detection-yolov5-traffic.git"
REF = "arena/01a0cfd1-object-detection-yolov5-traffi"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REF, REPO, str(ROOT)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
    check=True,
)
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU before the full training run.")

## Mount your private annotated dataset
This does not download a public dataset or invent annotations. Set DATASET to your
own directory in Drive. Never put Drive credentials or private footage in Git.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DATASET = Path("/content/drive/MyDrive/traffic-dataset")  # Edit this path.
if not DATASET.is_dir():
    raise FileNotFoundError("Supply the labeled dataset directory before continuing.")
import yaml

DATA = ROOT / "outputs/colab-data.yaml"
DATA.parent.mkdir(parents=True, exist_ok=True)
DATA.write_text(
    yaml.safe_dump(
        {
            "path": str(DATASET),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "Helmet", 1: "NoHelmet", 2: "LicensePlate"},
        }
    )
)
subprocess.run(
    [
        sys.executable,
        "training/train_yolov5.py",
        "--data",
        str(DATA),
        "--validate-only",
    ],
    cwd=ROOT,
    check=True,
)

## Train, evaluate and install the actual checkpoint
Adjust batch size for GPU memory. Training downloads the YOLOv5s initialization,
then learns from **your labels**. Only the resulting evaluated custom checkpoint
is installed; pretrained initialization alone is not the deliverable.
The command refuses to overwrite an existing models/best.pt unless explicitly authorized.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "training/train_yolov5.py",
        "--data",
        str(DATA),
        "--weights",
        "yolov5s.pt",
        "--epochs",
        "50",
        "--batch",
        "16",
        "--device",
        "0",
        "--install-model",
    ],
    cwd=ROOT,
    check=True,
)
reports = sorted((ROOT / "runs/train").glob("safecity-*/training_report.json"))
report = json.loads(reports[-1].read_text())
print(json.dumps(report, indent=2))

## Review real metrics and held-out examples
Inspect results.csv, loss plots, confusion matrices and validation outputs from
the paths printed above. There is no universal pass threshold baked into this
notebook; compare actual precision/recall/mAP with your internship requirements.
Use a separate held-out image/video, not the training examples, for the smoke test.


In [ ]:
IMAGE = DATASET / "heldout/example.jpg"  # Supply real held-out files.
VIDEO = DATASET / "heldout/example.mp4"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.verify_release",
        "--image",
        str(IMAGE),
        "--video",
        str(VIDEO),
    ],
    cwd=ROOT,
    check=True,
    env={**os.environ, "MODEL_PATH": str(ROOT / "models/best.pt")},
)

## Download the handover artifacts
Only run after evaluating the actual output. Keep private weights/data out of Git;
share a trusted artifact separately. The SHA-256 identifies the file, not its accuracy.
Restart the API with this best.pt and run the Postman/browser checks from README.


In [ ]:
import hashlib
from google.colab import files

BEST = ROOT / "models/best.pt"
with BEST.open("rb") as f:
    print("MODEL_SHA256:", hashlib.file_digest(f, "sha256").hexdigest())
files.download(str(BEST))
files.download(str(reports[-1]))